# Colab Fit Diagnostic Notebook

Diagnoses why Fit doesn't start on Google Colab.

Run each cell **in order** on Colab and share the output.

## 0. Install

In [ ]:
!pip install -q lizyml-widget lizyml[plots,tuning,calibration,explain]

## 1. Environment Info

In [ ]:
import sys, platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

import anywidget, ipywidgets, traitlets
print(f"anywidget: {anywidget.__version__}")
print(f"ipywidgets: {ipywidgets.__version__}")
print(f"traitlets: {traitlets.__version__}")

import lizyml_widget
print(f"lizyml_widget: {lizyml_widget.__version__}")

try:
    import google.colab
    print("google.colab: available")
except ImportError:
    print("google.colab: NOT available")

## 2. JS Colab Detection Test

This is the **critical test**. It checks whether our `isColab()` JS function returns `true`.

In [ ]:
from IPython.display import display, Javascript

display(Javascript("""
(function() {
    const results = {};

    // Test 1: Current detection method
    results.linkHrefColab = document.querySelector('link[href*="colab"]') !== null;

    // Test 2: window.google.colab
    results.windowGoogleColab = !!(window.google && window.google.colab);

    // Test 3: colab-specific elements
    results.colabNotebookEl = document.querySelector('colab-notebook') !== null;
    results.colabToolbarEl = document.querySelector('colab-toolbar') !== null;

    // Test 4: body classes or IDs
    results.bodyId = document.body?.id || '(none)';
    results.bodyClasses = document.body?.className || '(none)';

    // Test 5: all <link> tags (href contains what?)
    const links = Array.from(document.querySelectorAll('link[rel="stylesheet"]'));
    results.stylesheetHrefs = links.map(l => l.href).filter(h => h);

    // Test 6: meta tags
    const metas = Array.from(document.querySelectorAll('meta'));
    results.metaTags = metas.map(m => ({name: m.name, content: m.content})).filter(m => m.name);

    // Test 7: user agent
    results.userAgent = navigator.userAgent;

    // Output
    const el = document.createElement('pre');
    el.id = 'colab-diag-results';
    el.style.cssText = 'background:#1e1e1e;color:#0f0;padding:12px;border-radius:8px;font-size:13px;max-height:600px;overflow:auto;';
    el.textContent = JSON.stringify(results, null, 2);
    element.appendChild(el);
})();
"""))

## 3. BG Thread → JS Communication Test

Tests whether traitlet writes from a background thread reach JS on Colab.

In [ ]:
import anywidget
import traitlets
import threading
import time

class DiagWidget(anywidget.AnyWidget):
    _esm = """
    export function render({ model, el }) {
        const log = [];
        function addLog(msg) {
            log.push(`[${new Date().toISOString().slice(11,19)}] ${msg}`);
            el.innerHTML = `<pre style="background:#1e1e1e;color:#0f0;padding:12px;border-radius:8px;font-size:13px;max-height:400px;overflow:auto;">${log.join('\\n')}</pre>`;
        }

        // isColab check (same as production code)
        const linkColab = document.querySelector('link[href*="colab"]') !== null;
        const googleColab = !!(window.google && window.google.colab);
        addLog(`isColab detection: link[href*=colab]=${linkColab}, window.google.colab=${googleColab}`);

        // Listen for traitlet changes
        let traitletReceived = false;
        model.on('change:counter', () => {
            traitletReceived = true;
            addLog(`TRAITLET change:counter = ${model.get('counter')}`);
        });

        // Listen for msg:custom
        let msgReceived = false;
        model.on('msg:custom', (msg) => {
            msgReceived = true;
            addLog(`MSG:CUSTOM received: ${JSON.stringify(msg)}`);
        });

        // After 5 seconds, report what was received
        setTimeout(() => {
            addLog('--- 5s Summary ---');
            addLog(`Traitlet from BG thread received: ${traitletReceived}`);
            addLog(`msg:custom from BG thread received: ${msgReceived}`);
            if (!traitletReceived && !msgReceived) {
                addLog('DIAGNOSIS: BG thread comm is BLOCKED (expected on Colab)');
            } else {
                addLog('DIAGNOSIS: BG thread comm is WORKING');
            }
        }, 5000);

        addLog('Waiting for BG thread writes...');
    }
    """
    counter = traitlets.Int(0).tag(sync=True)

    def start_bg_test(self):
        """Start a background thread that writes to traitlets and sends custom msgs."""
        def worker():
            time.sleep(1)
            # Write traitlet from BG thread
            self.counter = 1
            time.sleep(0.5)
            # Send custom message from BG thread
            try:
                self.send({"type": "bg_test", "value": 42})
            except Exception as e:
                print(f"send() from BG thread failed: {e}")
            time.sleep(0.5)
            self.counter = 2
        t = threading.Thread(target=worker, daemon=True)
        t.start()
        print("BG thread started. Watch the widget output above.")

w = DiagWidget()
display(w)

In [ ]:
# Run this AFTER the widget above is displayed
w.start_bg_test()

## 4. Poll Mechanism Test

Tests whether JS → Python → JS custom message round-trip works (the polling fallback).

In [ ]:
class PollDiagWidget(anywidget.AnyWidget):
    _esm = """
    export function render({ model, el }) {
        const log = [];
        function addLog(msg) {
            log.push(`[${new Date().toISOString().slice(11,19)}] ${msg}`);
            el.innerHTML = `<pre style="background:#1e1e1e;color:#0f0;padding:12px;border-radius:8px;font-size:13px;max-height:400px;overflow:auto;">${log.join('\\n')}</pre>`;
        }

        // Listen for reply
        model.on('msg:custom', (msg) => {
            addLog(`REPLY received: ${JSON.stringify(msg)}`);
        });

        // Send poll request
        addLog('Sending poll request via model.send()...');
        model.send({ type: 'poll' });

        // Send another after 2s
        setTimeout(() => {
            addLog('Sending 2nd poll request...');
            model.send({ type: 'poll' });
        }, 2000);

        setTimeout(() => {
            if (log.length <= 3) {
                addLog('DIAGNOSIS: Poll round-trip FAILED — model.send() or self.send() broken');
            } else {
                addLog('DIAGNOSIS: Poll round-trip WORKING');
            }
        }, 5000);
    }
    """
    poll_count = traitlets.Int(0).tag(sync=True)

    def _handle_custom_msg(self, content, buffers):
        if content.get('type') == 'poll':
            self.poll_count += 1
            self.send({'type': 'poll_reply', 'count': self.poll_count})
        else:
            super()._handle_custom_msg(content, buffers)

pw = PollDiagWidget()
display(pw)

## 5. Actual LizyWidget Fit Test

Minimal Fit test with logging.

In [ ]:
import logging
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('lizyml_widget').setLevel(logging.DEBUG)

import pandas as pd
from sklearn.datasets import load_iris
from lizyml_widget import LizyWidget

iris = load_iris(as_frame=True)
df = pd.concat([iris.data, iris.target.rename('target')], axis=1)
print(f"Data shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

w = LizyWidget()
w.load(df, target='target')
print(f"Status after load: {w.status}")
print(f"Config: {dict(w.config)}")
display(w)

In [ ]:
# After clicking Fit in the widget UI above, run this cell to check Python-side state:
import time
print(f"Status: {w.status}")
print(f"Job type: {w.job_type}")
print(f"Progress: {dict(w.progress)}")
print(f"Elapsed: {w.elapsed_sec}")
print(f"Error: {dict(w.error)}")
if w._job_thread:
    print(f"Job thread alive: {w._job_thread.is_alive()}")
else:
    print("Job thread: None (never started)")
print(f"Fit summary: {dict(w.fit_summary)}")

## 6. Direct Python Fit (bypass widget)

Verifies that the ML backend itself works on Colab.

In [ ]:
# Direct LizyML fit (no widget, no threads)
try:
    import lizyml
    model = lizyml.LizyModel()
    model.fit(df.drop('target', axis=1), df['target'])
    print(f"Direct fit SUCCESS. Score: {model.score(df.drop('target', axis=1), df['target']):.4f}")
except Exception as e:
    print(f"Direct fit FAILED: {e}")